# Toy Model Extraction Demo (2D)

This notebook demonstrates **model extraction** (a.k.a. stealing) in a simplified setting:

- A **target model** acts as a black-box oracle `f(x)`.
- An attacker queries points `x_i` and collects outputs `y_i`.
- A **surrogate** is trained on the extracted dataset and evaluated by **fidelity** (agreement with the target).

No downloads required.


In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
def make_rings(n, noise=0.10, r0=1.0, r1=2.0):
    n0 = n // 2
    n1 = n - n0
    t0 = 2 * np.pi * np.random.rand(n0).astype(np.float32)
    t1 = 2 * np.pi * np.random.rand(n1).astype(np.float32)
    x0 = np.stack([r0 * np.cos(t0), r0 * np.sin(t0)], axis=1).astype(np.float32)
    x1 = np.stack([r1 * np.cos(t1), r1 * np.sin(t1)], axis=1).astype(np.float32)
    x0 += noise * np.random.randn(n0, 2).astype(np.float32)
    x1 += noise * np.random.randn(n1, 2).astype(np.float32)
    X = np.concatenate([x0, x1], axis=0)  # (n, 2)
    y = np.concatenate([
        np.zeros((n0, 1), dtype=np.float32),
        np.ones((n1, 1), dtype=np.float32),
    ], axis=0)  # (n, 1)
    perm = np.random.permutation(n)
    return X[perm], y[perm]


N_target_train = 1500
X_t_np, y_t_np = make_rings(N_target_train, noise=0.10)

# Tensors: X_t: (N_target_train, 2), y_t: (N_target_train, 1)
X_t = torch.tensor(X_t_np, device=device)
y_t = torch.tensor(y_t_np, device=device)

X_t.shape, y_t.shape

In [ ]:
class TargetMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        # x: (N, 2) -> logits: (N, 1)
        return self.net(x)


target = TargetMLP().to(device)
opt = torch.optim.Adam(target.parameters(), lr=1e-2)
loss_fn = nn.BCEWithLogitsLoss()

for e in range(1500):
    logits = target(X_t)  # (N_target_train, 1)
    loss = loss_fn(logits, y_t)  # scalar
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()

target.eval()
print("Target trained.")

In [ ]:
@torch.no_grad()
def oracle_labels(Xq):
    """Black-box oracle.
    Xq: (Q, 2)
    Returns yq: (Q, 1) in {0,1}
    """
    logits = target(Xq)  # (Q, 1)
    probs = torch.sigmoid(logits)  # (Q, 1)
    return (probs >= 0.5).float()


# Evaluation set for fidelity (random points in a box)
Q_eval = 6000
X_eval = (torch.rand((Q_eval, 2), device=device) * 6.0) - 3.0  # (Q_eval, 2) in [-3,3]^2
y_eval_oracle = oracle_labels(X_eval)  # (Q_eval, 1)
X_eval.shape, y_eval_oracle.shape

In [ ]:
class Surrogate(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 32),
            nn.Tanh(),
            nn.Linear(32, 32),
            nn.Tanh(),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        # x: (N, 2) -> logits: (N, 1)
        return self.net(x)


def train_surrogate(X_ext, y_ext, *, epochs=700, lr=1e-2):
    """X_ext: (Q,2), y_ext: (Q,1)"""
    s = Surrogate().to(device)
    opt = torch.optim.Adam(s.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()
    s.train()
    for _ in range(epochs):
        logits = s(X_ext)  # (Q, 1)
        loss = loss_fn(logits, y_ext)  # scalar
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
    s.eval()
    return s


@torch.no_grad()
def fidelity(surrogate, X_eval, y_eval_oracle):
    """Agreement with oracle on X_eval."""
    probs = torch.sigmoid(surrogate(X_eval))  # (Q_eval, 1)
    y_hat = (probs >= 0.5).float()  # (Q_eval, 1)
    return float((y_hat == y_eval_oracle).float().mean().item())

In [ ]:
budgets = [20, 50, 100, 200, 400, 800, 1500]
fids = []

for Q in budgets:
    # Query points: Xq: (Q, 2)
    Xq = (torch.rand((Q, 2), device=device) * 6.0) - 3.0
    yq = oracle_labels(Xq)  # (Q, 1)

    s = train_surrogate(Xq, yq, epochs=700, lr=1e-2)
    fid = fidelity(s, X_eval, y_eval_oracle)
    fids.append(fid)
    print(f"Q={Q:4d}  fidelity={fid:.3f}")

plt.figure(figsize=(7, 4))
plt.plot(budgets, fids, marker="o")
plt.xscale("log")
plt.ylim(0.5, 1.0)
plt.grid(True, alpha=0.3)
plt.title("Model extraction: fidelity vs query budget")
plt.xlabel("# queries (log scale)")
plt.ylabel("fidelity (agreement with oracle)")
plt.tight_layout()
plt.show()
